# Capybara Framework — Read the Docs Recovery Workbench

Target site: **https://capybara-framework.readthedocs.io**

This notebook performs three coordinated recovery jobs:

1. **Public mirror** — sequentially mirrors the published Read the Docs site, including HTML, CSS, JavaScript, graphics, `_sources` files, `_downloads` files, local video/audio, and downloadable artifacts.
2. **Authenticated Read the Docs snapshot** — opens Read the Docs Community in Selenium so you can log in normally, then saves project/dashboard pages plus read-only API v3 metadata, versions, redirects, environment-variable metadata, builds, and `build detail ?expand=config`.
3. **Repository reconstruction** — restores `_sources/*.rst.txt` / `.md.txt` back to source files, follows exposed `include` / `literalinclude` dependencies, maps hashed `_downloads` and `_images` files back to the relative paths named by the source, and creates a candidate GitHub-ready Sphinx tree.

## Politeness / safety

- No parallel requests.
- Every notebook-initiated HTTP request, RTD API request, or Selenium navigation waits a **random 1–5 seconds first**.
- Dashboard capture is **read-only**: no form submission, build triggering, deletion, deactivation, resync, or settings changes.
- Secret-looking form values are redacted in saved dashboard snapshots.
- External video services such as YouTube/Vimeo are inventoried but are **not downloaded by default**.
- Chrome itself may fetch CSS/JS dependencies when Selenium navigates to a dashboard page; the notebook cannot individually delay each browser-internal dependent request. Public-site mirroring itself is fully sequential and throttled per fetched URL.

The original `conf.py`, `.readthedocs.yaml`, `_templates`, and unreferenced repository files may not have been published. The notebook therefore labels generated build files as **recovery candidates**, not byte-for-byte originals.


In [ ]:
# Cell 1 — Configuration and imports

from __future__ import annotations

import csv
import hashlib
import html
import json
import mimetypes
import random
import re
import shutil
import time
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path, PurePosixPath
from typing import Optional
from urllib.error import HTTPError
from urllib.parse import unquote, urljoin, urlparse, urlunparse
from urllib.request import Request, build_opener, HTTPRedirectHandler

PROJECT_SLUG = "capybara-framework"
PUBLIC_BASE = "https://capybara-framework.readthedocs.io/"
RTD_APP_BASE = "https://app.readthedocs.org"

OUTPUT_ROOT = Path.cwd() / "capybara_readthedocs_recovery"
PUBLIC_MIRROR = OUTPUT_ROOT / "public_mirror"
ADMIN_SNAPSHOT = OUTPUT_ROOT / "admin_snapshot"
API_SNAPSHOT = OUTPUT_ROOT / "api_snapshot"
RECOVERED_REPO = OUTPUT_ROOT / "recovered_repo"
MANIFESTS = OUTPUT_ROOT / "manifests"

MIN_DELAY_SECONDS = 1.0
MAX_DELAY_SECONDS = 5.0

MAX_PUBLIC_URLS = 10000
MAX_ADMIN_PAGES = 300
MAX_BUILD_DETAILS = 100
MAX_SOURCE_INCLUDE_PROBES = 1000
MAX_RESPONSE_BYTES = 2 * 1024 * 1024 * 1024

FOLLOW_ALL_PUBLIC_VERSIONS = True
DOWNLOAD_EXTERNAL_MEDIA = False
SAVE_ADMIN_SCREENSHOTS = True
RESUME_EXISTING = True

for folder in (OUTPUT_ROOT, PUBLIC_MIRROR, ADMIN_SNAPSHOT, API_SNAPSHOT, RECOVERED_REPO, MANIFESTS):
    folder.mkdir(parents=True, exist_ok=True)

print("Recovery root:", OUTPUT_ROOT.resolve())
print(f"Throttle: random {MIN_DELAY_SECONDS:.1f}–{MAX_DELAY_SECONDS:.1f} seconds")


In [ ]:
# Cell 2 — Shared throttled downloader and link discovery

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def polite_wait(label="request"):
    seconds = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
    print(f"  ⏳ {label}: {seconds:.2f}s")
    time.sleep(seconds)
    return seconds

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def safe_piece(s, max_len=140):
    s = unquote(str(s or "")).strip()
    s = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", s)
    s = re.sub(r"\s+", " ", s).strip(" .") or "_"
    if len(s) > max_len:
        h = hashlib.sha1(s.encode("utf-8", "replace")).hexdigest()[:10]
        s = s[:max_len-11] + "_" + h
    return s

IGNORED_SCHEMES = {"mailto","tel","javascript","data","blob","about"}

def normalize_url(url, base=None):
    if not url:
        return None
    url = html.unescape(str(url).strip())
    if base:
        url = urljoin(base, url)
    p = urlparse(url)
    if p.scheme.lower() in IGNORED_SCHEMES or p.scheme.lower() not in {"http","https"}:
        return None
    path = re.sub(r"/{2,}", "/", p.path or "/")
    # Preserve queries only for API URLs. This prevents search/query crawl explosions.
    query = p.query if "/api/" in path else ""
    return urlunparse((p.scheme.lower(), p.netloc.lower(), path, p.params, query, ""))

def is_docs_host(url):
    return urlparse(url).netloc.lower() == urlparse(PUBLIC_BASE).netloc.lower()

def is_project_download(url):
    p = urlparse(url)
    return (
        p.netloc.lower() == urlparse(RTD_APP_BASE).netloc.lower()
        and f"/projects/{PROJECT_SLUG}/" in p.path
        and "/downloads/" in p.path
    )

MEDIA_EXTS = {
    ".png",".jpg",".jpeg",".gif",".svg",".webp",".ico",".bmp",".tif",".tiff",
    ".mp4",".webm",".mov",".m4v",".avi",".mp3",".wav",".ogg",".m4a",
    ".pdf",".epub",".zip",".7z",".tar",".gz",".tgz",
    ".ipynb",".py",".ps1",".sh",".bat",".cmd",
    ".doc",".docx",".xls",".xlsx",".ppt",".pptx",".csv",".tsv",".json",".xml",
    ".css",".js",".woff",".woff2",".ttf",".otf"
}

def likely_asset(url):
    path = urlparse(url).path.lower()
    return any(path.endswith(ext) for ext in MEDIA_EXTS)

def public_allowed(url):
    if is_docs_host(url) or is_project_download(url):
        return True
    return bool(DOWNLOAD_EXTERNAL_MEDIA and likely_asset(url))

def local_path_for(root, url, content_type=""):
    p = urlparse(url)
    parts = [safe_piece(x) for x in unquote(p.path or "/").split("/") if x]
    if (p.path or "/").endswith("/") or not parts:
        parts.append("index.html")
    elif "." not in PurePosixPath(parts[-1]).name and "text/html" in (content_type or "").lower():
        parts.append("index.html")
    return root / safe_piece(p.netloc) / Path(*parts)

class Collector(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.urls = []
        self.styles = []
        self.in_style = False

    def handle_starttag(self, tag, attrs):
        d = dict(attrs)
        if tag.lower() == "style":
            self.in_style = True
        mapping = {
            "a": ["href"], "link":["href"], "script":["src"], "img":["src","srcset"],
            "source":["src","srcset"], "video":["src","poster"], "audio":["src"],
            "iframe":["src"], "embed":["src"], "object":["data"]
        }
        for attr in mapping.get(tag.lower(), []):
            v = d.get(attr)
            if not v:
                continue
            if attr == "srcset":
                for part in v.split(","):
                    if part.strip():
                        self.urls.append(part.strip().split()[0])
            else:
                self.urls.append(v)
        if d.get("style"):
            self.styles.append(d["style"])

    def handle_endtag(self, tag):
        if tag.lower() == "style":
            self.in_style = False

    def handle_data(self, data):
        if self.in_style:
            self.styles.append(data)

CSS_URL_RE = re.compile(r'''url\(\s*(['"]?)(.*?)\1\s*\)''', re.I|re.S)

def css_urls(text):
    out = []
    for _, u in CSS_URL_RE.findall(text or ""):
        u = u.strip()
        if u and not u.lower().startswith(("data:","#")):
            out.append(u)
    return out

def discover_urls(base_url, data, content_type):
    found = []
    ct = (content_type or "").lower()
    path = urlparse(base_url).path.lower()

    if "html" in ct or path.endswith((".html",".htm","/")):
        text = data.decode("utf-8","replace")
        p = Collector()
        try:
            p.feed(text)
        except Exception:
            pass
        raw_urls = list(p.urls)
        for block in p.styles:
            raw_urls.extend(css_urls(block))
        for raw in raw_urls:
            u = normalize_url(raw, base_url)
            if u:
                found.append(u)

    elif "css" in ct or path.endswith(".css"):
        text = data.decode("utf-8","replace")
        for raw in css_urls(text):
            u = normalize_url(raw, base_url)
            if u:
                found.append(u)

    elif "xml" in ct or path.endswith((".xml",".rss",".atom")):
        text = data.decode("utf-8","replace")
        for raw in re.findall(r"<loc>\s*(.*?)\s*</loc>", text, re.I|re.S):
            u = normalize_url(html.unescape(raw), base_url)
            if u:
                found.append(u)

    return list(dict.fromkeys(found))

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status: int
    content_type: str
    content_length: int
    sha256: str
    local_path: Optional[str]
    error: Optional[str] = None

class Fetcher:
    def __init__(self, root):
        self.root = Path(root)
        self.opener = build_opener(HTTPRedirectHandler())
        self.records = []
        self.ua = "CapybaraFramework-RTD-Recovery/1.0 (owner backup; sequential 1-5s throttle)"

    def fetch(self, url, save=True):
        url = normalize_url(url)
        if not url:
            return FetchResult("", "", 0, "", 0, "", None, "invalid URL"), None

        # Resume only when a deterministic existing path is available.
        provisional = local_path_for(self.root, url, mimetypes.guess_type(urlparse(url).path)[0] or "")
        if save and RESUME_EXISTING and provisional.exists() and provisional.stat().st_size > 0:
            data = provisional.read_bytes()
            r = FetchResult(url,url,200,mimetypes.guess_type(str(provisional))[0] or "",len(data),sha256_bytes(data),str(provisional))
            self.records.append({**r.__dict__, "timestamp":now_utc(), "resumed":True})
            print("  ↪ existing")
            return r, data

        polite_wait("HTTP GET")
        try:
            req = Request(url, headers={"User-Agent":self.ua, "Accept":"*/*"})
            with self.opener.open(req, timeout=120) as resp:
                data = resp.read(MAX_RESPONSE_BYTES + 1)
                if len(data) > MAX_RESPONSE_BYTES:
                    raise RuntimeError("response exceeds configured safety cap")
                final_url = normalize_url(resp.geturl()) or url
                ct = resp.headers.get("Content-Type","").split(";")[0].strip().lower()
                path = None
                if save:
                    path = local_path_for(self.root, final_url, ct)
                    path.parent.mkdir(parents=True, exist_ok=True)
                    path.write_bytes(data)
                r = FetchResult(url,final_url,getattr(resp,"status",200),ct,len(data),sha256_bytes(data),str(path) if path else None)
        except HTTPError as e:
            r = FetchResult(url,normalize_url(e.geturl()) or url,int(e.code),e.headers.get_content_type() if e.headers else "",0,"",None,f"HTTPError: {e}")
            data = None
        except Exception as e:
            r = FetchResult(url,url,0,"",0,"",None,f"{type(e).__name__}: {e}")
            data = None

        self.records.append({**r.__dict__, "timestamp":now_utc(), "resumed":False})
        return r, data

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

print("Downloader utilities ready.")


In [ ]:
# Cell 3 — Mirror the complete public Read the Docs site

def mirror_public_site(start_url=PUBLIC_BASE, max_urls=MAX_PUBLIC_URLS):
    fetcher = Fetcher(PUBLIC_MIRROR)

    seeds = [
        normalize_url(start_url),
        normalize_url(urljoin(start_url, "en/latest/")),
        normalize_url(urljoin(start_url, "robots.txt")),
        normalize_url(urljoin(start_url, "sitemap.xml")),
    ]

    # Try both current/legacy offline-download shapes. 404s are simply recorded.
    for fmt in ("htmlzip","pdf","epub"):
        seeds.append(normalize_url(f"https://{PROJECT_SLUG}.readthedocs.io/_/downloads/{fmt}/{PROJECT_SLUG}/latest/"))
        seeds.append(normalize_url(f"https://{PROJECT_SLUG}.readthedocs.io/_/downloads/en/latest/{fmt}/"))

    q = deque(u for u in seeds if u)
    queued = set(q)
    seen = set()
    external = set()
    sources = set()
    downloads = set()
    media = set()
    errors = []

    while q and len(seen) < max_urls:
        url = q.popleft()
        queued.discard(url)
        if url in seen:
            continue
        seen.add(url)

        print(f"\nPUBLIC [{len(seen):04d}] {url}")
        result, data = fetcher.fetch(url, save=True)

        if result.error or result.status >= 400 or data is None:
            print("  ✗", result.status, result.error or "")
            errors.append(result.__dict__)
            continue

        print(f"  ✓ {result.status} {result.content_type or 'unknown'} {result.content_length:,} bytes")
        final_url = result.final_url or url
        pl = urlparse(final_url).path.lower()

        if "/_sources/" in pl:
            sources.add(final_url)
        if "/_downloads/" in pl or is_project_download(final_url):
            downloads.add(final_url)
        if likely_asset(final_url):
            media.add(final_url)

        for u in discover_urls(final_url, data, result.content_type):
            upl = urlparse(u).path.lower()
            if "/_sources/" in upl:
                sources.add(u)
            if "/_downloads/" in upl or is_project_download(u):
                downloads.add(u)
            if likely_asset(u):
                media.add(u)

            if public_allowed(u):
                # Deliberately avoid account/actions and search-query traps.
                if any(x in upl for x in ("/accounts/","/logout/","/delete/","/remove/","/deactivate/","/revoke/","/trigger/")):
                    continue
                if is_docs_host(u) and not FOLLOW_ALL_PUBLIC_VERSIONS:
                    if "/en/latest/" not in upl and "/_/" not in upl:
                        continue
                if u not in seen and u not in queued:
                    q.append(u)
                    queued.add(u)
            else:
                external.add(u)

    summary = {
        "finished_at": now_utc(),
        "fetched_or_attempted": len(seen),
        "remaining_queue": len(q),
        "errors": len(errors),
        "source_urls": len(sources),
        "download_urls": len(downloads),
        "media_asset_urls": len(media),
        "external_urls_inventoried": len(external),
        "throttle_seconds": [MIN_DELAY_SECONDS, MAX_DELAY_SECONDS],
    }

    write_json(MANIFESTS/"public_fetch_records.json", fetcher.records)
    write_json(MANIFESTS/"public_summary.json", summary)
    write_json(MANIFESTS/"public_source_urls.json", sorted(sources))
    write_json(MANIFESTS/"public_download_urls.json", sorted(downloads))
    write_json(MANIFESTS/"public_media_urls.json", sorted(media))
    write_json(MANIFESTS/"external_urls_inventory.json", sorted(external))
    write_json(MANIFESTS/"public_errors.json", errors)

    print("\nPUBLIC MIRROR COMPLETE")
    print(json.dumps(summary, indent=2))
    return summary

# Run when ready:
# public_summary = mirror_public_site()


## Authenticated dashboard/API capture

Run the next cell to open Chrome. **Log in manually** in that Chrome window; do not place credentials in the notebook.

The authenticated capture uses:

- Selenium-rendered dashboard/settings pages, text, screenshots, links, and redacted form/control values.
- Read the Docs API v3 through the already-authenticated browser session.
- Build detail calls using `?expand=config`, which are particularly important for reconstructing the vanished `.readthedocs.yaml` behavior.

All calls are GET/read-only operations.


In [ ]:
# Cell 4 — Start Selenium, log in manually

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def start_rtd_browser():
    opts = Options()
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-notifications")
    opts.add_argument("--disable-popup-blocking")
    driver = webdriver.Chrome(options=opts)
    driver.set_page_load_timeout(120)
    polite_wait("Selenium navigation")
    driver.get(f"{RTD_APP_BASE}/dashboard/")
    print("Chrome opened. Log into Read the Docs manually.")
    print("After you can see your dashboard, return here and run Cell 5.")
    return driver

# Run:
# driver = start_rtd_browser()


In [ ]:
# Cell 5 — Authenticated read-only API + dashboard snapshot

SECRET_RE = re.compile(r"(password|passwd|secret|token|csrf|auth|api[_-]?key|private|credential)", re.I)
DANGEROUS_RE = re.compile(r"/(?:delete|remove|deactivate|revoke|destroy|logout|sync-versions|trigger|wipe|resync)(?:/|$)", re.I)

def browser_get(driver, url):
    polite_wait("RTD API GET")
    script = r'''
        const url = arguments[0];
        const done = arguments[arguments.length - 1];
        fetch(url, {method:'GET', credentials:'include', headers:{'Accept':'application/json, text/plain, */*'}})
          .then(async r => done({ok:r.ok,status:r.status,url:r.url,contentType:r.headers.get('content-type')||'',text:await r.text()}))
          .catch(e => done({ok:false,status:0,url:url,contentType:'',text:'',error:String(e)}));
    '''
    return driver.execute_async_script(script, url)

def browser_get_json(driver, url):
    meta = browser_get(driver, url)
    try:
        obj = json.loads(meta.get("text",""))
    except Exception:
        obj = None
    return meta, obj

def api_all(driver, url, label):
    rows = []
    page = 0
    while url:
        page += 1
        if url.startswith("/"):
            url = urljoin(RTD_APP_BASE, url)
        print(f"{label} API page {page}: {url}")
        meta, obj = browser_get_json(driver, url)
        if not meta.get("ok") or not isinstance(obj, dict):
            print("  ✗", meta.get("status"), meta.get("error",""))
            break
        if isinstance(obj.get("results"), list):
            rows.extend(obj["results"])
        url = urljoin(RTD_APP_BASE, obj["next"]) if obj.get("next") else None
    return rows

def redact_control(x):
    x = dict(x)
    signature = " ".join(str(x.get(k,"")) for k in ("name","id","type","label"))
    if x.get("type","").lower() == "password" or SECRET_RE.search(signature):
        if x.get("value") not in ("",None,False,[]):
            x["value"] = "[REDACTED]"
    return x

def page_snapshot(driver):
    js = r'''
    const controls = Array.from(document.querySelectorAll('input,select,textarea')).map(el => {
      let value;
      if (el.tagName === 'SELECT') value = Array.from(el.selectedOptions).map(o=>o.value);
      else if (el.type === 'checkbox' || el.type === 'radio') value = el.checked;
      else value = el.value;
      let label = '';
      if (el.id) {
        const l = document.querySelector('label[for="' + CSS.escape(el.id) + '"]');
        if (l) label = (l.innerText || l.textContent || '').trim();
      }
      return {tag:el.tagName.toLowerCase(),type:el.type||'',name:el.name||'',id:el.id||'',label:label,value:value,disabled:!!el.disabled,required:!!el.required};
    });
    const links = Array.from(document.querySelectorAll('a[href]')).map(a => ({href:a.href,text:(a.innerText||a.textContent||'').trim(),title:a.title||''}));
    return {url:location.href,title:document.title,bodyText:document.body?document.body.innerText:'',links:links,controls:controls};
    '''
    snap = driver.execute_script(js)
    snap["controls"] = [redact_control(x) for x in snap.get("controls",[])]
    snap["captured_at"] = now_utc()
    return snap

def admin_safe(url, anchor_text=""):
    u = normalize_url(url)
    if not u:
        return False
    p = urlparse(u)
    if p.netloc.lower() != urlparse(RTD_APP_BASE).netloc.lower():
        return False
    if DANGEROUS_RE.search(p.path):
        return False
    if re.search(r"\b(delete|remove|deactivate|revoke|trigger|rebuild|wipe|destroy)\b", anchor_text or "", re.I):
        return False
    return PROJECT_SLUG in p.path or f"/dashboard/{PROJECT_SLUG}/" in p.path

def dashboard_basename(url, n):
    p = urlparse(url)
    bits = [safe_piece(x,80) for x in p.path.strip("/").split("/") if x] or ["root"]
    return ADMIN_SNAPSHOT/"pages"/f"{n:04d}__{'__'.join(bits)}"

def crawl_dashboard(driver, max_pages=MAX_ADMIN_PAGES):
    start = f"{RTD_APP_BASE}/projects/{PROJECT_SLUG}/"
    q = deque([start])
    queued = {normalize_url(start)}
    seen = set()
    records = []

    while q and len(seen) < max_pages:
        url = normalize_url(q.popleft())
        if not url or url in seen:
            continue
        seen.add(url)
        print(f"\nADMIN [{len(seen):03d}] {url}")
        polite_wait("Selenium navigation")
        try:
            driver.get(url)
            time.sleep(0.8)
            snap = page_snapshot(driver)
        except Exception as e:
            records.append({"url":url,"error":str(e),"captured_at":now_utc()})
            print("  ✗", e)
            continue

        base = dashboard_basename(driver.current_url, len(seen))
        base.parent.mkdir(parents=True, exist_ok=True)
        base.with_suffix(".html").write_text(driver.page_source, encoding="utf-8", errors="replace")
        base.with_suffix(".txt").write_text(snap.get("bodyText",""), encoding="utf-8", errors="replace")
        write_json(base.with_suffix(".json"), snap)
        if SAVE_ADMIN_SCREENSHOTS:
            try:
                driver.save_screenshot(str(base.with_suffix(".png")))
            except Exception:
                pass

        records.append({"url":driver.current_url,"title":driver.title,"snapshot":str(base.with_suffix(".json")),"captured_at":now_utc()})

        for link in snap.get("links",[]):
            href = normalize_url(link.get("href",""))
            if href and admin_safe(href, link.get("text","")) and href not in seen and href not in queued:
                q.append(href)
                queued.add(href)

        print("  ✓", driver.title)

    write_json(MANIFESTS/"admin_pages.json", records)
    return {"captured":len(records),"seen":len(seen),"remaining":len(q),"finished_at":now_utc()}

def capture_api(driver):
    root = f"{RTD_APP_BASE}/api/v3/projects/{PROJECT_SLUG}/"
    print("\nPROJECT API")
    meta, project = browser_get_json(driver, root)
    write_json(API_SNAPSHOT/"project_response_meta.json", meta)
    if not isinstance(project, dict):
        raise RuntimeError(f"Project API unavailable (HTTP {meta.get('status')}). Is the Selenium browser logged in?")
    write_json(API_SNAPSHOT/"project.json", project)

    endpoints = {
        "versions": root+"versions/?limit=100",
        "builds": root+"builds/?limit=100",
        "redirects": root+"redirects/?limit=100",
        "environmentvariables": root+"environmentvariables/?limit=100",
        "subprojects": root+"subprojects/?limit=100",
        "translations": root+"translations/?limit=100",
    }
    captured = {"project":project}

    for name, url in endpoints.items():
        print("\n"+name.upper())
        rows = api_all(driver, url, name)
        captured[name] = rows
        write_json(API_SNAPSHOT/f"{name}.json", rows)

    details = []
    builds = captured.get("builds") or []
    for i, b in enumerate(builds[:MAX_BUILD_DETAILS],1):
        if not isinstance(b,dict) or not b.get("id"):
            continue
        bid = b["id"]
        print(f"\nBUILD DETAIL {i}/{min(len(builds),MAX_BUILD_DETAILS)}: {bid}")
        meta, obj = browser_get_json(driver, f"{root}builds/{bid}/?expand=config")
        rec = {"meta":meta,"data":obj}
        details.append(rec)
        write_json(API_SNAPSHOT/"build_details"/f"build_{bid}.json", rec)

    write_json(API_SNAPSHOT/"build_details_index.json", details)

    # Download any PDF/ePub/htmlzip URLs advertised by version metadata.
    dl_fetcher = Fetcher(PUBLIC_MIRROR)
    advertised = []
    for v in captured.get("versions") or []:
        if not isinstance(v,dict):
            continue
        for fmt, url in (v.get("downloads") or {}).items():
            if not url:
                continue
            item = {"version":v.get("slug"),"format":fmt,"url":url}
            print(f"\nOFFLINE DOWNLOAD: {item['version']} / {fmt}")
            result, _ = dl_fetcher.fetch(url, save=True)
            item["result"] = result.__dict__
            advertised.append(item)

    write_json(API_SNAPSHOT/"advertised_downloads.json", advertised)
    write_json(MANIFESTS/"api_download_fetch_records.json", dl_fetcher.records)

    summary = {
        "versions":len(captured.get("versions") or []),
        "builds":len(captured.get("builds") or []),
        "build_details":len(details),
        "redirects":len(captured.get("redirects") or []),
        "environmentvariables":len(captured.get("environmentvariables") or []),
        "subprojects":len(captured.get("subprojects") or []),
        "translations":len(captured.get("translations") or []),
        "offline_downloads":len(advertised),
        "captured_at":now_utc(),
    }
    write_json(API_SNAPSHOT/"summary.json", summary)
    return summary

def capture_authenticated_rtd(driver, crawl_admin=True):
    api_summary = capture_api(driver)
    admin_summary = crawl_dashboard(driver) if crawl_admin else None
    out = {"api":api_summary,"dashboard":admin_summary,"captured_at":now_utc()}
    write_json(MANIFESTS/"authenticated_capture_summary.json", out)
    print(json.dumps(out, indent=2))
    return out

# After manually logging in:
# auth_summary = capture_authenticated_rtd(driver, crawl_admin=True)


In [ ]:
# Cell 6 — Reconstruct a candidate Sphinx/GitHub repository

RST_INCLUDE_RE = re.compile(r"^\s*\.\.\s+(include|literalinclude)::\s+(.+?)\s*$", re.I|re.M)
RST_IMAGE_RE = re.compile(r"^\s*\.\.\s+(?:image|figure)::\s+(.+?)\s*$", re.I|re.M)
RST_DOWNLOAD_RE = re.compile(r":download:`(?:[^`<>]*?<)?([^`<>]+?)(?:>)?`", re.I)
MD_ASSET_RE = re.compile(r"!?\[[^\]]*\]\(([^)\s]+)(?:\s+['\"][^'\"]*['\"])?\)")

def load_records():
    out = []
    for name in ("public_fetch_records.json","api_download_fetch_records.json","include_probe_fetch_records.json"):
        p = MANIFESTS/name
        if p.exists():
            out.extend(json.loads(p.read_text(encoding="utf-8")))
    return out

def url_path_index():
    idx = {}
    for r in load_records():
        if r.get("status") == 200 and r.get("local_path"):
            p = Path(r["local_path"])
            if p.exists():
                idx[r.get("final_url") or r.get("requested_url")] = p
    return idx

def source_repo_rel(url):
    p = urlparse(url)
    marker = "/_sources/"
    if marker not in p.path:
        return None
    rel = unquote(p.path.split(marker,1)[1])
    if rel.endswith(".txt"):
        rel = rel[:-4]
    rel = rel.lstrip("/")
    if not rel or ".." in PurePosixPath(rel).parts:
        return None
    return Path("docs")/Path(*PurePosixPath(rel).parts)

def clean_target(raw):
    raw = html.unescape(str(raw).strip()).strip("'\"")
    if not raw or raw.startswith(("http://","https://","#","mailto:","data:")):
        return None
    if "<" in raw and raw.endswith(">"):
        raw = raw.rsplit("<",1)[1][:-1].strip()
    raw = raw.split("#",1)[0].split("?",1)[0].strip()
    return raw or None

def resolve_repo_rel(source_rel, target):
    target = clean_target(target)
    if not target:
        return None
    base = PurePosixPath(source_rel.as_posix()).parent
    t = PurePosixPath(target)
    combined = PurePosixPath(str(t).lstrip("/")) if t.is_absolute() else base/t
    stack = []
    for part in combined.parts:
        if part in ("","."):
            continue
        if part == "..":
            if not stack:
                return None
            stack.pop()
        else:
            stack.append(part)
    return Path(*stack) if stack else None

def recover_direct_sources(idx):
    rows = []
    for url, src in sorted(idx.items()):
        rel = source_repo_rel(url)
        if not rel:
            continue
        dest = RECOVERED_REPO/rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src,dest)
        rows.append({"source_url":url,"repo_path":str(rel),"method":"_sources direct","sha256":sha256_bytes(dest.read_bytes())})
    return rows

def probe_includes(source_rows):
    fetcher = Fetcher(PUBLIC_MIRROR)
    q = deque(source_rows)
    attempted = set()
    added = []
    probes = 0

    while q and probes < MAX_SOURCE_INCLUDE_PROBES:
        rec = q.popleft()
        repo_rel = Path(rec["repo_path"])
        p = RECOVERED_REPO/repo_rel
        if not p.exists():
            continue
        text = p.read_text(encoding="utf-8",errors="replace")
        for m in RST_INCLUDE_RE.finditer(text):
            target = clean_target(m.group(2))
            if not target:
                continue
            dest_rel = resolve_repo_rel(repo_rel,target)
            if not dest_rel or (RECOVERED_REPO/dest_rel).exists():
                continue
            candidate = normalize_url(urljoin(rec["source_url"],target))
            if not candidate:
                continue
            if not candidate.endswith(".txt"):
                candidate += ".txt"
            if candidate in attempted:
                continue
            attempted.add(candidate)
            probes += 1
            print(f"\nSOURCE INCLUDE PROBE [{probes}] {candidate}")
            result,data = fetcher.fetch(candidate,save=True)
            if result.status == 200 and data:
                dest = RECOVERED_REPO/dest_rel
                dest.parent.mkdir(parents=True,exist_ok=True)
                dest.write_bytes(data)
                row = {"source_url":candidate,"repo_path":str(dest_rel),"method":"include/literalinclude probe","sha256":sha256_bytes(data)}
                added.append(row)
                q.append(row)
                print("  ✓ recovered:",dest_rel)

    write_json(MANIFESTS/"include_probe_fetch_records.json",fetcher.records)
    return added

def basename_index(idx):
    out = defaultdict(list)
    for url,p in idx.items():
        base = unquote(PurePosixPath(urlparse(url).path).name).lower()
        if base:
            out[base].append((url,p))
    return out

def extract_refs(repo_rel,text):
    refs = []
    for m in RST_IMAGE_RE.finditer(text):
        refs.append(("image",m.group(1)))
    for m in RST_DOWNLOAD_RE.finditer(text):
        refs.append(("download",m.group(1)))
    for m in MD_ASSET_RE.finditer(text):
        target = m.group(1)
        if any(target.lower().split("?",1)[0].endswith(ext) for ext in MEDIA_EXTS):
            refs.append(("markdown_asset",target))

    seen = set()
    out = []
    for kind,target in refs:
        rel = resolve_repo_rel(repo_rel,target)
        if rel and (kind,str(rel)) not in seen:
            seen.add((kind,str(rel)))
            out.append({"kind":kind,"target":clean_target(target),"repo_path":str(rel)})
    return out

def map_assets(source_rows,idx):
    bidx = basename_index(idx)
    recovered, unresolved = [], []

    for rec in source_rows:
        src_rel = Path(rec["repo_path"])
        p = RECOVERED_REPO/src_rel
        if not p.exists():
            continue
        text = p.read_text(encoding="utf-8",errors="replace")

        for ref in extract_refs(src_rel,text):
            dest_rel = Path(ref["repo_path"])
            dest = RECOVERED_REPO/dest_rel
            if dest.exists():
                continue
            candidates = bidx.get(dest_rel.name.lower(),[])
            scored = []
            for url,path in candidates:
                score = 0
                upl = urlparse(url).path.lower()
                if ref["kind"] == "download" and "/_downloads/" in upl:
                    score += 10
                if ref["kind"] == "image" and "/_images/" in upl:
                    score += 10
                scored.append((score,url,path))
            scored.sort(reverse=True,key=lambda x:x[0])

            chosen = None
            if len(scored) == 1:
                chosen = scored[0]
            elif len(scored) > 1 and scored[0][0] > scored[1][0]:
                chosen = scored[0]

            if chosen:
                _,url,path = chosen
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,dest)
                recovered.append({"referenced_by":str(src_rel),**ref,"published_url":url})
            else:
                unresolved.append({"referenced_by":str(src_rel),**ref,"candidate_urls":[u for u,_ in candidates]})

    return recovered,unresolved

def preserve_extras(idx):
    count = 0
    for url,src in idx.items():
        path = urlparse(url).path
        marker = next((m for m in ("/_static/","/_images/","/_downloads/") if m in path),None)
        if not marker:
            continue
        rel = path.split(marker,1)[1].lstrip("/")
        dest = RECOVERED_REPO/"recovery_extras"/marker.strip("/")/Path(*PurePosixPath(unquote(rel)).parts)
        if not dest.exists():
            dest.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(src,dest)
            count += 1
    return count

def newest_build_with_config():
    p = API_SNAPSHOT/"build_details_index.json"
    if not p.exists():
        return None
    rows = json.loads(p.read_text(encoding="utf-8"))
    builds = [r.get("data") for r in rows if isinstance(r.get("data"),dict) and r["data"].get("config")]
    builds.sort(key=lambda x:x.get("created") or "",reverse=True)
    return builds[0] if builds else None

def infer_extensions():
    exts = set()
    docs = RECOVERED_REPO/"docs"
    for p in docs.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".rst",".md",".txt"}:
            continue
        text = p.read_text(encoding="utf-8",errors="replace")
        if p.suffix.lower()==".md" or "myst_parser" in text:
            exts.add("myst_parser")
        if ".. automodule::" in text or ".. autoclass::" in text:
            exts.add("sphinx.ext.autodoc")
        if ".. graphviz::" in text:
            exts.add("sphinx.ext.graphviz")
        if ".. mermaid::" in text:
            exts.add("sphinxcontrib.mermaid")
    return sorted(exts)

def write_scaffold():
    docs = RECOVERED_REPO/"docs"
    docs.mkdir(parents=True,exist_ok=True)
    hist = newest_build_with_config()
    cfg = hist.get("config",{}) if hist else {}
    exts = infer_extensions()

    conf = docs/"conf.py"
    if not conf.exists():
        ext_lines = ",\n    ".join(repr(x) for x in exts)
        conf.write_text(f'''# RECOVERY-GENERATED CANDIDATE conf.py
# Compare against api_snapshot/build_details and the surviving public site.

project = "Capybara Framework"
author = "Capybara Framework Team"

extensions = [
    {ext_lines}
]

source_suffix = {{".rst":"restructuredtext",".md":"markdown"}}
master_doc = "index"

# Conservative recovery default; change if the old build evidence identifies another theme.
html_theme = "sphinx_rtd_theme"

templates_path = ["_templates"]
exclude_patterns = ["_build","Thumbs.db",".DS_Store"]
html_static_path = ["_static"] if (Path(__file__).parent/"_static").exists() else []
''',encoding="utf-8")

    reqs = ["sphinx","sphinx-rtd-theme"]
    if "myst_parser" in exts:
        reqs.append("myst-parser")
    if "sphinxcontrib.mermaid" in exts:
        reqs.append("sphinxcontrib-mermaid")
    req = docs/"requirements.txt"
    if not req.exists():
        req.write_text("\n".join(reqs)+"\n",encoding="utf-8")

    pyver = "3.12"
    osname = "ubuntu-24.04"
    formats = []
    if isinstance(cfg,dict):
        b = cfg.get("build") or {}
        if isinstance(b,dict):
            if b.get("os"):
                osname = str(b["os"])
            if isinstance(b.get("tools"),dict) and b["tools"].get("python"):
                pyver = str(b["tools"]["python"])
        if isinstance(cfg.get("python"),dict) and cfg["python"].get("version"):
            pyver = str(cfg["python"]["version"])
        if isinstance(cfg.get("formats"),list):
            formats = [str(x) for x in cfg["formats"] if str(x).lower() in {"pdf","epub"}]

    yaml_lines = [
        "# RECOVERY-GENERATED CANDIDATE .readthedocs.yaml",
        "version: 2","",
        "build:",f'  os: "{osname}"',"  tools:",f'    python: "{pyver}"',"",
        "sphinx:","  configuration: docs/conf.py","",
        "python:","  install:","    - requirements: docs/requirements.txt"
    ]
    if formats:
        yaml_lines += ["","formats:"] + [f"  - {x}" for x in formats]

    yml = RECOVERED_REPO/".readthedocs.yaml"
    if not yml.exists():
        yml.write_text("\n".join(yaml_lines)+"\n",encoding="utf-8")

    gi = RECOVERED_REPO/".gitignore"
    if not gi.exists():
        gi.write_text("_build/\ndocs/_build/\n.venv/\n__pycache__/\n*.pyc\n.DS_Store\nThumbs.db\n",encoding="utf-8")

    return {
        "historical_build_id": hist.get("id") if hist else None,
        "historical_config": cfg,
        "inferred_extensions": exts,
        "requirements": reqs,
        "candidate_python": pyver,
        "candidate_os": osname,
        "candidate_formats": formats
    }

def write_report(sources,assets,unresolved,extras,scaffold):
    lines = [
        "# Capybara Framework Recovery Report","",
        f"Generated: `{now_utc()}`","",
        "## Recovered",
        f"- Sphinx/source files: **{len(sources)}**",
        f"- Referenced assets/downloads mapped to source-relative paths: **{len(assets)}**",
        f"- Published `_static`, `_images`, `_downloads` files preserved under `recovery_extras/`: **{extras}**",
        f"- Unresolved references: **{len(unresolved)}**","",
        "## Build reconstruction",
        f"- Historical build used as configuration evidence: `{scaffold.get('historical_build_id')}`",
        f"- Candidate Python: `{scaffold.get('candidate_python')}`",
        f"- Candidate OS: `{scaffold.get('candidate_os')}`",
        f"- Inferred extensions: `{', '.join(scaffold.get('inferred_extensions') or []) or '(none)'}`","",
        "The generated `conf.py` and `.readthedocs.yaml` are candidates, not byte-for-byte originals. "
        "Review `../api_snapshot/build_details/`, `../admin_snapshot/`, and this report before reconnecting Read the Docs.","",
        "## Unresolved references",""
    ]
    if not unresolved:
        lines.append("None detected by the mapper.")
    else:
        for x in unresolved:
            lines += [
                f"### `{x.get('repo_path')}`",
                f"- Referenced by: `{x.get('referenced_by')}`",
                f"- Kind: `{x.get('kind')}`",
                f"- Original target: `{x.get('target')}`",
                f"- Published candidates: `{len(x.get('candidate_urls') or [])}`",""
            ]
    (RECOVERED_REPO/"RECOVERY_REPORT.md").write_text("\n".join(lines)+"\n",encoding="utf-8")

def reconstruct_repo():
    idx = url_path_index()
    if not idx:
        raise RuntimeError("No mirror records found. Run mirror_public_site() first.")

    print("Restoring directly published _sources...")
    direct = recover_direct_sources(idx)
    print("Direct sources:",len(direct))

    print("Probing include/literalinclude dependencies...")
    included = probe_includes(direct)
    sources = direct + included

    # Reload index after include probes.
    idx = url_path_index()

    print("Mapping images/downloads back to source-relative paths...")
    assets,unresolved = map_assets(sources,idx)

    print("Preserving published static/image/download evidence...")
    extras = preserve_extras(idx)

    print("Creating candidate build scaffolding...")
    scaffold = write_scaffold()

    write_json(MANIFESTS/"recovered_sources.json",sources)
    write_json(MANIFESTS/"recovered_assets.json",assets)
    write_json(MANIFESTS/"unresolved_references.json",unresolved)
    write_json(MANIFESTS/"scaffold_inference.json",scaffold)
    write_report(sources,assets,unresolved,extras,scaffold)

    summary = {
        "repo":str(RECOVERED_REPO.resolve()),
        "sources":len(sources),
        "mapped_assets":len(assets),
        "unresolved":len(unresolved),
        "recovery_extras":extras,
        "historical_build_hint":scaffold.get("historical_build_id"),
        "finished_at":now_utc()
    }
    write_json(MANIFESTS/"reconstruction_summary.json",summary)
    print(json.dumps(summary,indent=2))
    print("Report:",(RECOVERED_REPO/"RECOVERY_REPORT.md").resolve())
    return summary

# Run after the public mirror:
# reconstruction_summary = reconstruct_repo()


In [ ]:
# Cell 7 — Inventories and ZIP archives

def inventory(root):
    rows = []
    for p in sorted(Path(root).rglob("*")):
        if p.is_file():
            try:
                data = p.read_bytes()
                rows.append({"path":str(p.relative_to(root)),"bytes":len(data),"sha256":sha256_bytes(data),"error":""})
            except Exception as e:
                rows.append({"path":str(p.relative_to(root)),"bytes":None,"sha256":"","error":str(e)})
    return rows

def finalize_recovery():
    summary = {}
    for name,root in [
        ("public_mirror",PUBLIC_MIRROR),
        ("admin_snapshot",ADMIN_SNAPSHOT),
        ("api_snapshot",API_SNAPSHOT),
        ("recovered_repo",RECOVERED_REPO)
    ]:
        rows = inventory(root)
        summary[name] = {"files":len(rows),"bytes":sum(r["bytes"] or 0 for r in rows)}
        write_json(MANIFESTS/f"{name}_file_inventory.json",rows)
        with (MANIFESTS/f"{name}_file_inventory.csv").open("w",newline="",encoding="utf-8") as f:
            w = csv.DictWriter(f,fieldnames=["path","bytes","sha256","error"])
            w.writeheader()
            w.writerows(rows)

    repo_zip = OUTPUT_ROOT/"capybara-framework-recovered-repo"
    shutil.make_archive(str(repo_zip),"zip",root_dir=RECOVERED_REPO)

    bundle_zip = OUTPUT_ROOT.parent/"capybara_readthedocs_recovery_bundle"
    shutil.make_archive(str(bundle_zip),"zip",root_dir=OUTPUT_ROOT)

    write_json(MANIFESTS/"final_inventory_summary.json",summary)
    result = {
        "inventory":summary,
        "repo_zip":str(repo_zip.with_suffix(".zip").resolve()),
        "bundle_zip":str(bundle_zip.with_suffix(".zip").resolve())
    }
    print(json.dumps(result,indent=2))
    return result

# Run last:
# final_summary = finalize_recovery()


# Recommended run order

### 1. Mirror the surviving public documentation first

```python
public_summary = mirror_public_site()
```

This is the highest-priority preservation step.

### 2. Reconstruct what is already recoverable

```python
reconstruction_summary = reconstruct_repo()
```

Because your pages expose **View page source**, this should restore the `.rst`/`.md` page tree and many relative asset/download locations.

### 3. Capture Read the Docs dashboard/build configuration

```python
driver = start_rtd_browser()
```

Log in manually in Chrome, then:

```python
auth_summary = capture_authenticated_rtd(driver, crawl_admin=True)
```

The `api_snapshot/build_details/` directory is particularly important because each build detail is requested with `expand=config`.

### 4. Re-run reconstruction after authenticated capture

If you want the candidate `.readthedocs.yaml` regenerated using the historical build configuration, delete or rename the generated candidate `.readthedocs.yaml` and `docs/conf.py`, then run:

```python
reconstruction_summary = reconstruct_repo()
```

### 5. Create checksummed inventories and ZIPs

```python
final_summary = finalize_recovery()
```

Important outputs:

- `capybara_readthedocs_recovery/public_mirror/`
- `capybara_readthedocs_recovery/admin_snapshot/`
- `capybara_readthedocs_recovery/api_snapshot/`
- `capybara_readthedocs_recovery/recovered_repo/`
- `capybara_readthedocs_recovery/manifests/`
- `capybara_readthedocs_recovery/capybara-framework-recovered-repo.zip`
- `capybara_readthedocs_recovery_bundle.zip`

Before reconnecting the existing Read the Docs project to a new GitHub repository, review `recovered_repo/RECOVERY_REPORT.md` and successfully run a local Sphinx build.
